In [5]:
!pip install chardet==3.0.4

import spacy
import re
from typing import List, Set, Dict
from pathlib import Path
import json

import chardet




class SkillsExtractor:
    def __init__(self):
        # Load English language model
        self.nlp = spacy.load("en_core_web_sm")

        # Common skill-related keywords
        self.skill_patterns = [
            r"ability to",
            r"proficient in",
            r"experience with",
            r"knowledge of",
            r"understanding of",
            r"expertise in",
            r"skilled in",
            r"competent in",
            r"familiar with",
            r"mastery of"
        ]

        # Define skill-related verbs
        self.skill_verbs = {
            "analyze", "develop", "manage", "create", "design", "implement",
            "evaluate", "research", "coordinate", "maintain", "operate",
            "perform", "plan", "program", "utilize", "write", "apply",
            "assess", "build", "calculate", "code", "communicate",
            "demonstrate", "document", "identify", "solve", "understand"
        }

    def extract_skills_from_text(self, text: str) -> Set[str]:
        """Extract skills from given text using NLP processing."""
        skills = set()
        doc = self.nlp(text)

        # Process each sentence
        for sent in doc.sents:
            # Look for skill patterns
            for pattern in self.skill_patterns:
                matches = re.finditer(pattern, sent.text.lower())
                for match in matches:
                    # Get the text following the pattern
                    end_pos = match.end()
                    remaining_text = sent.text[end_pos:].strip()
                    if remaining_text:
                        # Extract noun phrases following the pattern
                        remaining_doc = self.nlp(remaining_text)
                        for chunk in remaining_doc.noun_chunks:
                            skills.add(chunk.text.strip().lower())

            # Look for verb-based skills
            for token in sent:
                if token.lemma_.lower() in self.skill_verbs:
                    # Check if followed by direct object
                    if token.dep_ == "ROOT":
                        for child in token.children:
                            if child.dep_ in ("dobj", "pobj"):
                                skill = f"{token.lemma_} {child.text}".lower()
                                skills.add(skill)

        return skills

    def extract_skills_from_file(self, file_path: str) -> Dict[str, List[str]]:
        """Extract skills from a text file."""
        try:
            # Read the text file
            text = Path(file_path).read_text(encoding='cp1252')

            # Extract skills
            skills = self.extract_skills_from_text(text)

            # Organize results
            result = {
                "file": file_path,
                "skills": sorted(list(skills)),
                "count": len(skills)
            }

            return result

        except Exception as e:
            return {
                "file": file_path,
                "error": str(e),
                "skills": [],
                "count": 0
            }


def main():
    # Initialize the skills extractor
    extractor = SkillsExtractor()

    # Example usage
    file_path = "/syllabus.txt"  # Replace with your text file path

    # Extract skills
    results = extractor.extract_skills_from_file(file_path)

    # Print results in a formatted way
    print("\nSkills Extraction Results:")
    print("-" * 50)
    if "error" in results:
        print(f"Error processing file: {results['error']}")
    else:
        print(f"File: {results['file']}")
        print(f"Total skills found: {results['count']}")
        print("\nExtracted Skills:")
        for skill in results['skills']:
            print(f"- {skill}")

    # Save results to JSON file
    output_file = "extracted_skills.json"
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=2)
    print(f"\nResults saved to {output_file}")

if __name__ == "__main__":
    main()


Skills Extraction Results:
--------------------------------------------------
File: /syllabus.txt
Total skills found: 4

Extracted Skills:
- communicate this
- understand break
- understand ch
- understand monopoly

Results saved to extracted_skills.json
